# XGBoost Hyperparameter Optimization (V05)


## Objectives


### English

The objective of this notebook is to systematically optimize the XGBoost classifier through hyperparameter tuning and evaluate whether the optimized model can outperform the current Random Forest baseline.

The notebook explores the influence of the main XGBoost hyperparameters, applies a randomized hyperparameter search using cross-validation, and compares the optimized model against all previously evaluated classifiers.

This experiment represents the first systematic optimization stage of the project before proceeding to Monte Carlo World Cup simulations.

### Español

El objetivo de este cuaderno es optimizar sistemáticamente el clasificador XGBoost mediante el ajuste de hiperparámetros y evaluar si el modelo optimizado supera al clasificador de referencia actual, Random Forest.

El cuaderno explora la influencia de los principales hiperparámetros de XGBoost, aplica una búsqueda aleatoria de hiperparámetros mediante validación cruzada y compara el modelo optimizado con todos los clasificadores evaluados previamente.

Este experimento representa la primera etapa de optimización sistemática del proyecto antes de proceder a las simulaciones de la Copa Mundial de Montecarlo.

## Imports

In [ ]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier

from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Load Dataset

In [ ]:
match_vector = pd.read_csv("../data/processed/match_vector_v03.csv")

print(match_vector.shape)

match_vector.head()

## Utility Functions

From Notebook 09

In [ ]:
def prepare_train_test_data(
    df,
    target_col="target",
    train_year=2018,
    test_year=2022,
    drop_cols=None,
):
    """
    Prepare train and test datasets using World Cup editions.

    Parameters
    ----------
    df : pandas.DataFrame
        Match-level dataset.
    target_col : str
        Target column.
    train_year : int
        Year used for training.
    test_year : int
        Year used for testing.
    drop_cols : list, optional
        Additional columns to exclude from the feature matrix.

    Returns
    -------
    X_train, X_test, y_train, y_test
    """

    if drop_cols is None:
        drop_cols = [
            "match_id",
            "match_date",
            "kick_off",
            "year",
            "competition_stage",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            target_col,
        ]

    X = df.drop(columns=drop_cols)
    y = df[target_col]

    train_mask = df["year"] == train_year
    test_mask = df["year"] == test_year

    X_train = X.loc[train_mask]
    X_test = X.loc[test_mask]

    y_train = y.loc[train_mask]
    y_test = y.loc[test_mask]

    return X_train, X_test, y_train, y_test



In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    """
    Train and evaluate a classification model.

    Parameters
    ----------
    model : estimator
        Any sklearn-compatible classification model.
    X_train, X_test : pandas.DataFrame
        Training and testing feature matrices.
    y_train, y_test : pandas.Series
        Training and testing targets.

    Returns
    -------
    dict
        Dictionary containing accuracy, predictions,
        classification report, and confusion matrix.
    """

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    results = {
        "accuracy": accuracy_score(y_test, y_pred),
        "predictions": y_pred,
        "classification_report": classification_report(y_test, y_pred),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
    }

    return results



## Baseline XGBoost